# 🚀 Stage 3: TarDAL Joint Backbone & Head Fine-Tuning — Google Colab

This notebook provides the complete GPU-accelerated pipeline for **joint, end-to-end fine-tuning of both the TarDAL Generator backbone and object detection head** on the M3FD dataset (`strategy = 'fuse & detect'`).

### 🏷️ TarDAL Backbone Checkpoint Options:
1. **`dt` (Direct Training)** 🏆 *(Recommended baseline)*: High spatial & boundary feature retention.
2. **`stage3_gen_best.pt`** 🏆 *(Recommended recovery)*: Start from best adapted generator weights.
3. **`tt` (Task-oriented Training)**: Optimized detection-oriented features.
4. **`ct` (Cooperative Training)**: Joint human vision & target enhancement.

### 📁 Google Drive Path Mapping:
- **Project Code Path**: `/content/drive/MyDrive/FYP/code` (or `/content/drive/MyDrive/fyp/code`)
- **M3FD Dataset Archive**: `/content/drive/MyDrive/FYP/M3FD_Detection.zip`

### Step 1: Mount Google Drive & Verify GPU Acceleration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU hardware availability
!nvidia-smi

### Step 2: Install Project Dependencies

In [ ]:
%pip install -q kornia thop tabulate PyYAML tqdm opencv-python matplotlib pandas scipy ultralytics

### Step 3: Fast & Robust Environment Setup

In [ ]:
# @title ⚙️ Step 3: Fast & Robust Environment Setup
import sys
from pathlib import Path

# Dynamic project path resolution
cand_roots = [Path('/content/drive/MyDrive/FYP/code'), Path('/content/drive/MyDrive/fyp/code'), Path('/content/drive/MyDrive/code'), Path('/content/code'), Path.cwd()]
CODE_PATH = next((p for p in cand_roots if (p / 'scripts_AG').exists() or (p / 'TarDAL-main').exists()), Path.cwd())

for p in [CODE_PATH, CODE_PATH / 'scripts_AG', CODE_PATH / 'TarDAL-main']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from stage3_colab_setup import setup_stage3_environment
CODE_PATH, ds_root = setup_stage3_environment()


### Step 3.5: Run Pre-Flight Pipeline Smoke Test Suite

In [ ]:
# @title 🧪 Run Pre-Flight Pipeline Smoke Test Suite
smoke_cmd = f"python -W ignore {str(CODE_PATH / 'scripts_AG' / '99_smoke_test_stage3.py')}"
get_ipython().system(smoke_cmd)

### Step 4: Configure & Launch Joint Backbone & Detection Head Fine-Tuning

**Architecture**: Generator backbone (296K params) is UNFROZEN for task-driven adaptation.
Detection head (9.1M params) is PERMANENTLY FROZEN from Stage 2 `best.pt`.

In [ ]:
# @title 🎛️ Training Hyperparameters & Backbone Selection { run: "auto" }
BACKBONE_CHECKPOINT = "stage3_gen_best.pt" # @param ["stage3_gen_best.pt", "dt", "tt", "ct"]
EPOCHS = 30 # @param {type:"integer"}
BATCH_SIZE = 4 # @param {type:"integer"}
LEARNING_RATE = 5e-6 # @param {type:"number"}
SKIP_FUSION_IF_READY = False # @param {type:"boolean"}
FORCE_RESTART_FROM_EPOCH_1 = True # @param {type:"boolean"}

# Advanced Loss Bridge Normalization
W_FUSE = 1.0 # @param {type:"number"}
W_DET = 0.2 # @param {type:"number"}

print(f"Target Backbone Checkpoint : {BACKBONE_CHECKPOINT}")
print(f"Training Epochs           : {EPOCHS}")
print(f"Batch Size                : {BATCH_SIZE}")
print(f"Learning Rate             : {LEARNING_RATE}")
print(f"Skip Pre-Fusion           : {SKIP_FUSION_IF_READY}")
print(f"Force Restart (Epoch 1)   : {FORCE_RESTART_FROM_EPOCH_1}")
print(f"Loss Bridge Weights       : w_fuse={W_FUSE}, w_det={W_DET}")

# Launch Stage 3 joint end-to-end fine-tuning script with global warning suppression
train_cmd = (
    f"python -W ignore {str(CODE_PATH / 'scripts_AG' / '11_train_stage3_joint_end2end.py')}"
    f" --weights {BACKBONE_CHECKPOINT}"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --lr {LEARNING_RATE}"
    f" --w_fuse {W_FUSE}"
    f" --w_det {W_DET}"
)

if SKIP_FUSION_IF_READY:
    train_cmd += " --skip_fusion"
if FORCE_RESTART_FROM_EPOCH_1:
    train_cmd += " --force_restart"

# If user selected stage3_gen_best.pt, pass it explicitly as gen_ckpt
if BACKBONE_CHECKPOINT == "stage3_gen_best.pt":
    train_cmd += " --gen_ckpt /content/drive/MyDrive/FYP/code/checkpoints/stage3_gen_best.pt"

print(f"\n>>> {train_cmd}\n")
get_ipython().system(train_cmd)

### Step 5: Evaluate Fine-Tuned Model Performance (mAP@50 & mAP@50:95)

In [ ]:
# @title 📊 Stage 3 Evaluation & Benchmark Model Selection { run: "auto" }
GENERATOR_BACKBONE_CHECKPOINT = "stage3_gen_best.pt" # @param ["stage3_gen_best.pt", "stage3_gen_last.pt", "tardal_dt", "tardal-dt.pth", "tardal_ct", "tardal-ct.pth", "tardal_tt", "tardal-tt.pth", "dt", "ct", "tt"]
DETECTOR_HEAD_CHECKPOINT = "stage3_best.pt" # @param ["stage3_best.pt", "stage3_last.pt", "best.pt"]
DATASET_SPLIT = "val" # @param ["val", "test"]
RESET_EVALUATION = False # @param {type:"boolean"}

print(f"Generator Backbone Checkpoint : {GENERATOR_BACKBONE_CHECKPOINT}")
print(f"Detector Head Checkpoint      : {DETECTOR_HEAD_CHECKPOINT}")
print(f"Dataset Split Selection       : {DATASET_SPLIT}")
print(f"Reset Evaluation (Re-Fuse)    : {RESET_EVALUATION}")

eval_script = str(CODE_PATH / 'scripts_AG' / '12_eval_test_set_and_baselines.py')
eval_cmd = f"python -W ignore {eval_script} --gen_ckpt {GENERATOR_BACKBONE_CHECKPOINT} --det_ckpt {DETECTOR_HEAD_CHECKPOINT} --split {DATASET_SPLIT}"

if RESET_EVALUATION:
    eval_cmd += " --force_refuse"

get_ipython().system(eval_cmd)

### Step 6: Visualize Fine-Tuned Bounding Box Predictions

In [ ]:
# @title 🖼️ Display Sample Visualizations
viz_script = str(CODE_PATH / 'scripts_AG' / '04_visualize_results.py')
results_dir = str(CODE_PATH / 'runs' / 'stage3_joint_end2end')

get_ipython().system(f"python {viz_script} --run_dir {results_dir}")